# Decision Tree — CART (Classification And Regression Trees)

CART (Breiman et al., 1984) is the algorithm behind scikit-learn's trees. Key properties:

* **Always binary splits**: $x_j \le t$ vs $x_j > t$.
* **Classification** uses **Gini impurity**: $G(S) = 1 - \sum_k p_k^2$
* **Regression** uses **variance / MSE**: $\text{MSE}(S) = \frac{1}{|S|}\sum_{i\in S}(y_i - \bar{y}_S)^2$, leaves predict the mean.
* The best split minimises the weighted impurity of the children
  $$\frac{|S_L|}{|S|}I(S_L) + \frac{|S_R|}{|S|}I(S_R)$$
* **Cost-complexity pruning**: minimise $R_\alpha(T) = R(T) + \alpha|\text{leaves}(T)|$. Increasing $\alpha$ yields a nested
  sequence of subtrees found by repeatedly collapsing the "weakest link" — the node with smallest
  $$g(t) = \frac{R(t) - R(T_t)}{|\text{leaves}(T_t)| - 1}$$

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

import copy

In [ ]:
def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Implementation

Split search is vectorised: for each feature we sort the samples once and use cumulative class counts
(classification) or cumulative sums of $y$ and $y^2$ (regression) to evaluate *every* threshold in $O(n)$.

In [ ]:
class Node:
    def __init__(self, value, n, risk, feature=None, threshold=None, left=None, right=None):
        self.value = value          # class label or mean prediction at this node
        self.n = n                  # number of training samples reaching the node
        self.risk = risk            # R(t): node error as a leaf, weighted by n / N_total
        self.feature, self.threshold = feature, threshold
        self.left, self.right = left, right

    @property
    def is_leaf(self):
        return self.left is None


class CART:
    def __init__(self, task="classification", max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, max_features=None, random_state=None):
        assert task in ("classification", "regression")
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features          # used by Random Forests
        self.rng = np.random.RandomState(random_state)

    # ---------------------------------------------------------------- helpers
    def _node_value_and_risk(self, y):
        if self.task == "classification":
            counts = np.bincount(y, minlength=self.n_classes)
            return np.argmax(counts), (len(y) - counts.max()) / self.N
        return y.mean(), np.sum((y - y.mean()) ** 2) / self.N

    def _best_split(self, X, y):
        n, d = X.shape
        features = np.arange(d)
        if self.max_features is not None:
            features = self.rng.choice(d, self.max_features, replace=False)

        best_f, best_t, best_cost = None, None, np.inf
        nl = np.arange(1, n)                   # left child sizes for split after position i
        nr = n - nl
        for f in features:
            order = np.argsort(X[:, f], kind="mergesort")
            xs, ys = X[order, f], y[order]
            if self.task == "classification":
                onehot = np.eye(self.n_classes)[ys]
                left = np.cumsum(onehot, axis=0)[:-1]
                right = onehot.sum(axis=0) - left
                gini_l = 1 - np.sum((left / nl[:, None]) ** 2, axis=1)
                gini_r = 1 - np.sum((right / nr[:, None]) ** 2, axis=1)
                cost = (nl * gini_l + nr * gini_r) / n
            else:
                cs, cs2 = np.cumsum(ys)[:-1], np.cumsum(ys ** 2)[:-1]
                tot, tot2 = ys.sum(), (ys ** 2).sum()
                sse_l = cs2 - cs ** 2 / nl
                sse_r = (tot2 - cs2) - (tot - cs) ** 2 / nr
                cost = (sse_l + sse_r) / n
            valid = (xs[1:] != xs[:-1]) & (nl >= self.min_samples_leaf) & (nr >= self.min_samples_leaf)
            if not valid.any():
                continue
            cost = np.where(valid, cost, np.inf)
            i = np.argmin(cost)
            if cost[i] < best_cost:
                best_f, best_t, best_cost = f, (xs[i] + xs[i + 1]) / 2, cost[i]
        return best_f, best_t

    # ---------------------------------------------------------------- fitting
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        if self.task == "classification":
            self.classes_, y = np.unique(y, return_inverse=True)
            self.n_classes = len(self.classes_)
        else:
            y = np.asarray(y, dtype=float)
        self.N = len(y)
        self.root = self._build(X, y, 0)
        return self

    def _build(self, X, y, depth):
        value, risk = self._node_value_and_risk(y)
        node = Node(value, len(y), risk)
        pure = risk == 0
        if (pure or len(y) < self.min_samples_split
                or (self.max_depth is not None and depth >= self.max_depth)):
            return node
        f, t = self._best_split(X, y)
        if f is None:
            return node
        mask = X[:, f] <= t
        node.feature, node.threshold = f, t
        node.left = self._build(X[mask], y[mask], depth + 1)
        node.right = self._build(X[~mask], y[~mask], depth + 1)
        return node

    # ---------------------------------------------------------------- inference (vectorised)
    def _predict_node(self, node, X, idx, out):
        if node.is_leaf:
            out[idx] = node.value
            return
        mask = X[idx, node.feature] <= node.threshold
        self._predict_node(node.left, X, idx[mask], out)
        self._predict_node(node.right, X, idx[~mask], out)

    def predict(self, X, root=None):
        X = np.asarray(X, dtype=float)
        out = np.empty(len(X))
        self._predict_node(root or self.root, X, np.arange(len(X)), out)
        if self.task == "classification":
            return self.classes_[out.astype(int)]
        return out

    # ---------------------------------------------------------------- inspection
    @staticmethod
    def leaves(node):
        return 1 if node.is_leaf else CART.leaves(node.left) + CART.leaves(node.right)

    @staticmethod
    def depth(node):
        return 0 if node.is_leaf else 1 + max(CART.depth(node.left), CART.depth(node.right))

    def print_tree(self, node=None, indent="", names=None):
        node = node or self.root
        if node.is_leaf:
            v = self.classes_[node.value] if self.task == "classification" else round(node.value, 3)
            print(f"{indent}-> {v}  (n={node.n})")
            return
        name = names[node.feature] if names else f"x{node.feature}"
        print(f"{indent}{name} <= {node.threshold:.3f}")
        self.print_tree(node.left, indent + "  |  ", names)
        print(f"{indent}{name} >  {node.threshold:.3f}")
        self.print_tree(node.right, indent + "  |  ", names)

## Gini impurity vs entropy

In [ ]:
p = np.linspace(0, 1, 200)
gini = 1 - p**2 - (1 - p)**2
with np.errstate(divide="ignore", invalid="ignore"):
    ent = np.nan_to_num(-p * np.log2(p) - (1 - p) * np.log2(1 - p))
misclass = 1 - np.maximum(p, 1 - p)
plt.plot(p, gini, label="Gini"); plt.plot(p, ent / 2, label="Entropy / 2"); plt.plot(p, misclass, label="Misclassification")
plt.xlabel("P(class 1)"); plt.ylabel("impurity"); plt.legend(); plt.title("Impurity measures"); plt.show()

## Classification on two moons

In [ ]:
X, y = make_moons(n_samples=500, noise=0.3, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, d in zip(axes, [2, 5, None]):
    clf = CART(max_depth=d).fit(X_train, y_train)
    plot_decision_boundary(clf, X_train, y_train, ax=ax,
        title=f"max_depth={d}: leaves={CART.leaves(clf.root)}\n"
              f"train={accuracy_score(y_train, clf.predict(X_train)):.2f} "
              f"test={accuracy_score(y_test, clf.predict(X_test)):.2f}")
plt.tight_layout(); plt.show()

CART(max_depth=2).fit(X_train, y_train).print_tree()

## Cost-complexity (weakest-link) pruning

Grow the full tree, then repeatedly collapse the internal node with the smallest $g(t)$.
Each collapse gives the next tree in the pruning path and the $\alpha$ at which it becomes optimal.

In [ ]:
def subtree_risk(node):
    return node.risk if node.is_leaf else subtree_risk(node.left) + subtree_risk(node.right)


def internal_nodes(node):
    if node.is_leaf:
        return []
    return [node] + internal_nodes(node.left) + internal_nodes(node.right)


def cost_complexity_path(root):
    tree = copy.deepcopy(root)
    alphas, trees = [0.0], [copy.deepcopy(tree)]
    while not tree.is_leaf:
        best_g, weakest = np.inf, None
        for node in internal_nodes(tree):
            g = (node.risk - subtree_risk(node)) / (CART.leaves(node) - 1)
            if g < best_g:
                best_g, weakest = g, node
        weakest.left = weakest.right = None           # collapse into a leaf
        alphas.append(max(best_g, 0.0))
        trees.append(copy.deepcopy(tree))
    return np.array(alphas), trees


full = CART().fit(X_train, y_train)
alphas, trees = cost_complexity_path(full.root)
train_acc = [accuracy_score(y_train, full.predict(X_train, t)) for t in trees]
test_acc = [accuracy_score(y_test, full.predict(X_test, t)) for t in trees]
n_leaves = [CART.leaves(t) for t in trees]

best = int(np.argmax(test_acc))
print(f"Full tree: {n_leaves[0]} leaves, test acc {test_acc[0]:.3f}")
print(f"Best pruned tree: alpha={alphas[best]:.4f}, {n_leaves[best]} leaves, test acc {test_acc[best]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].step(alphas, train_acc, where="post", label="train")
axes[0].step(alphas, test_acc, where="post", label="test")
axes[0].set_xlabel("alpha"); axes[0].set_ylabel("accuracy"); axes[0].legend(); axes[0].set_title("Accuracy vs alpha")
axes[1].step(alphas, n_leaves, where="post"); axes[1].set_yscale("log")
axes[1].set_xlabel("alpha"); axes[1].set_ylabel("# leaves"); axes[1].set_title("Tree size vs alpha")
plt.tight_layout(); plt.show()

In [ ]:
class _Pruned:                       # tiny wrapper so plot_decision_boundary can use a pruned root
    def __init__(self, model, root): self.model, self.root = model, root
    def predict(self, X): return self.model.predict(X, self.root)

plot_decision_boundary(_Pruned(full, trees[best]), X_train, y_train,
                       f"Pruned CART (alpha={alphas[best]:.4f}, {n_leaves[best]} leaves)")
plt.show()

## Regression tree

In [ ]:
x = np.sort(np.random.uniform(0, 5, 200))
y_reg = np.sin(x) + np.random.normal(0, 0.15, len(x))
Xr = x.reshape(-1, 1)
grid = np.linspace(0, 5, 500).reshape(-1, 1)

plt.scatter(x, y_reg, s=12, c="gray", label="data")
for d in [1, 3, 6]:
    reg = CART(task="regression", max_depth=d).fit(Xr, y_reg)
    plt.plot(grid, reg.predict(grid), lw=2, label=f"max_depth={d}")
plt.legend(); plt.title("CART regression: piecewise-constant fits"); plt.show()

## Comparison of the three classic tree algorithms

| | ID3 | C4.5 | CART |
|---|---|---|---|
| Split criterion | Information gain | Gain ratio | Gini (classif.) / MSE (regr.) |
| Split type | Multi-way | Multi-way (categorical), binary (numeric) | Always binary |
| Feature types | Categorical | Categorical + continuous | Continuous (categoricals encoded) |
| Regression | No | No | Yes |
| Pruning | None | Pessimistic error pruning | Cost-complexity pruning |